In [ ]:
#@title 📦 Kaggle Competition Loader { display-mode: "form" }
#@markdown ---
#@markdown ### How to use
#@markdown 1. Run this cell (code stays hidden — double-click the title to peek).
#@markdown 2. When prompted, paste the **full Kaggle competition URL**
#@markdown    (e.g. `https://www.kaggle.com/c/titanic`) — or just the slug.
#@markdown 3. Make sure you're authenticated with `kagglehub` (it'll prompt on first run).
#@markdown 4. After it finishes, your data is available as:
#@markdown    - `data['train.csv']`, `data['test.csv']`, etc. → pandas DataFrames
#@markdown    - `data['some_file.ext']` → file path (non-CSV files)
#@markdown    - `data_dir` → the local folder containing everything
#@markdown ---

import os
import re
import shutil
import kagglehub
import pandas as pd


def _extract_slug(text: str) -> str:
    """Pulls the competition slug out of a full URL or accepts a bare slug."""
    text = text.strip().rstrip('/')
    match = re.search(r'kaggle\.com/(?:c|competitions)/([^/?#]+)', text)
    if match:
        return match.group(1)
    if "kaggle.com" in text:
        return text.split('/')[-1]
    return text


def fetch_kaggle_competition(comp_url: str = None, copy_to_local: bool = True):
    """
    Downloads a Kaggle competition dataset via kagglehub and loads all CSVs.

    Args:
        comp_url: Full Kaggle competition URL or slug. If None, you'll be prompted.
        copy_to_local: If True, copies files out of the kagglehub cache into ./<slug>/

    Returns:
        data: dict mapping filename -> DataFrame (for .csv) or filepath (other files)
        working_path: directory containing the dataset files
    """
    comp_input = comp_url or input("Paste Kaggle competition URL (or slug): ").strip()
    comp_id = _extract_slug(comp_input)

    if not comp_id:
        raise ValueError("Could not parse a competition slug from that input.")

    print(f"\n[1/3] Authenticating & downloading: '{comp_id}'...")
    kagglehub.login()
    cache_path = kagglehub.competition_download(comp_id)

    target_dir = os.path.join(os.getcwd(), comp_id)
    if copy_to_local:
        if os.path.exists(target_dir):
            shutil.rmtree(target_dir)
        shutil.copytree(cache_path, target_dir)
        working_path = target_dir
    else:
        working_path = cache_path

    print(f"[2/3] Files localized to: {working_path}")

    data = {}
    print("\n[3/3] Inspecting dataset contents:")
    for root, _, filenames in os.walk(working_path):
        for fname in sorted(filenames):
            fpath = os.path.join(root, fname)
            if fname.endswith('.csv'):
                data[fname] = pd.read_csv(fpath)
                print(f"  • {fname:30s} shape={data[fname].shape} -> data['{fname}']")
            else:
                data[fname] = fpath
                print(f"  • {fname:30s} (path saved) -> data['{fname}']")

    return data, working_path


# ==========================================
# RUN — paste your competition URL when asked
# ==========================================
data, data_dir = fetch_kaggle_competition()
print("\nAvailable keys:", list(data.keys()))

In [ ]:
#@title Submit predictions to Kaggle { display-mode: "form" }
#@markdown Reads Kaggle credentials from Colab secrets, validates each
#@markdown `submission_*.csv` against `sample_submission.csv`, then asks
#@markdown for a per-file y/n confirmation before submitting.
#@markdown
#@markdown ⚠️ **Most Kaggle competitions cap you at 5 submissions/day.**
#@markdown Check the "My Submissions" tab on the competition page for
#@markdown your remaining count before confirming each file — a 400/403
#@markdown error with no other obvious cause usually means you hit the cap.
#@markdown
#@markdown Assumes the Kaggle Competition Loader cell has already run
#@markdown (uses its `data` and `data_dir` globals).

from google.colab import userdata
import os, glob, pandas as pd

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_LEGACY_KEY')

COMP = os.path.basename(data_dir)   # slug taken from the loader cell, no re-entry needed
sample = data['sample_submission.csv']

for path in sorted(glob.glob("submission_*.csv")):
    sub = pd.read_csv(path)
    ok = (sub.shape == sample.shape
          and list(sub.columns) == list(sample.columns)
          and sub['id'].astype(int).equals(sample['id'].astype(int))
          and sub.isna().sum().sum() == 0)
    print(f"\n{path}: {'OK' if ok else 'MISMATCH'} — shape {sub.shape}, dtypes {dict(sub.dtypes)}")
    if not ok:
        print("  Skipping — fix the file before submitting.")
        continue
    if input(f"Submit {path}? (y/n): ").strip().lower() == 'y':
        msg = input("  Submission message: ").strip() or path
        os.system(f'kaggle competitions submit -c {COMP} -f {path} -m "{msg}"')